It is clear that there are a few compounds that are throwing the model off. We currently have many jobs running in parallel, with a join mp_20 and deepmind training dataset and 20u restriction. The scripts are set up to save compounds that throw errors as well as compounds that return infinite losses. However, it is a little tricky to pinpoint the compounds that are causing the issues since the entire batch is saved. A good first approach might be to look for repeats in the data 

In [1]:
import pandas as pd
import numpy as np 
import torch 
import ast 
import os

In [2]:
directory = '/home/gridsan/tmackey/cdvae/data/mp_20_dm/train_errors'

In [3]:
def load_pt_files(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.pt')]
    list_of_tensors = []

    for file in files:
        file_path = os.path.join(directory, file)
        tensors = torch.load(file_path)      
        list_of_tensors.append(tensors)
        
    return list_of_tensors

loaded_tensors = load_pt_files(directory)


In [4]:
loaded_tensors_batches = [loaded_tensor[0] for loaded_tensor in loaded_tensors]

In [5]:
loaded_tensors_xrd_ints = [loaded_tensor[1] for loaded_tensor in loaded_tensors]

In [6]:
from collections import Counter

# Assuming 'tensors' is your list of 79 256x256 tensors
# Flatten each tensor to get a 2D array (256, 256) -> (256, 256)
flattened_tensors = [tensor.view(tensor.size(0), -1) for tensor in loaded_tensors_xrd_ints]

# Concatenate all 2D arrays to get a single array
all_data_points = torch.cat(flattened_tensors, dim=0)

# Convert tensor rows to tuple for hashability
data_points_as_tuples = [tuple(row.tolist()) for row in all_data_points]

# Count occurrences of each data point
data_point_counts = Counter(data_points_as_tuples)

# Find the number of duplicates
duplicates = sum(1 for count in data_point_counts.values() if count > 1)

print(f"Number of duplicate data points: {duplicates}")


Number of duplicate data points: 41608


In [7]:
import torch
from collections import Counter

# Assuming 'tensors' is your list of 79 256x256 tensors
# Flatten each tensor to get a 2D array (256, 256) -> (256, 256)
flattened_tensors = [tensor.view(tensor.size(0), -1) for tensor in loaded_tensors_xrd_ints]

# Concatenate all 2D arrays to get a single array
all_data_points = torch.cat(flattened_tensors, dim=0)

# Convert tensor rows to tuple for hashability
data_points_as_tuples = [tuple(row.tolist()) for row in all_data_points]

# Count occurrences of each data point
data_point_counts = Counter(data_points_as_tuples)

# Find and print duplicate data points and their counts
duplicates = {dp: count for dp, count in data_point_counts.items() if count > 1}

# To keep only unique keys with the highest value among duplicates
unique_highest_value_duplicates = {}
for key, value in duplicates.items():
    if key not in unique_highest_value_duplicates or unique_highest_value_duplicates[key] < value:
        unique_highest_value_duplicates[key] = value

# Now unique_highest_value_duplicates contains unique keys with their highest values


In [8]:

# Sort the duplicates by count
sorted_duplicates = sorted(unique_highest_value_duplicates.items(), key=lambda x: x[1], reverse=True)

# Remove duplicates within the duplicates dictionary
unique_sorted_duplicates = dict(sorted_duplicates)


In [9]:
len(unique_sorted_duplicates)

41608

In [22]:
print(f"Duplicate Data Points and Counts:")
import time
for data_point, count in unique_sorted_duplicates.items():
    print(f"Count: {count}")
    time.sleep(1)

Duplicate Data Points and Counts:
Count: 363
Count: 271
Count: 65
Count: 25
Count: 23
Count: 23
Count: 23
Count: 23


KeyboardInterrupt: 

In [11]:
train_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20_dm/train.csv")

/state/partition1/slurm_tmp/24598142.0.0/ipykernel_3896495/3868518509.py:1: DtypeWarning: Columns (7,9,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20_dm/train.csv")


In [23]:
filtered_keys = {k: v for k, v in unique_sorted_duplicates.items() if v > 50}

# Identify rows in train_df where 'xrd_peak_intensities' matches the filtered keys
matching_rows = train_df[train_df['xrd_peak_intensities'].isin(filtered_keys.keys())]

In [28]:
type(filtered_keys)

dict

In [25]:
list_of_keys = [list(key) for key in list(filtered_keys.keys())]

In [13]:
xrd_intensities = torch.load("/home/gridsan/tmackey/cdvae/data/mp_20_dm/train_xrd_peak_intensities_dict.pt")

In [29]:
# Convert tensor values in the dictionary to lists
dict_as_lists = {key: value.tolist()[0] for key, value in xrd_intensities.items()}

# Filter the dictionary to keep only entries that match an entry in search_list
#filtered_dict = {key: value for key, value in dict_as_lists.items() if value in list_of_keys}
filtered_dict = {key: filtered_keys[tuple(value)] for key, value in dict_as_lists.items() if value in list_of_keys}

# 'filtered_dict' now contains the key-value pairs matching the search criteria

In [31]:
filtered_dict

{'train_not_true_mat_id_70762': 65,
 'train_not_true_mat_id_146154': 363,
 'train_not_true_mat_id_177708': 271}

In [88]:
problem_compounds = train_df[train_df['material_id'].isin(list(filtered_dict.keys()))]

In [92]:
problem_compounds['cif']

26408     # generated using pymatgen\ndata_Sm2Sb\n_symme...
97894     # generated using pymatgen\ndata_La9Cu2Pb6O\n_...
173283    # generated using pymatgen\ndata_U2FeCo2P3\n_s...
204503    # generated using pymatgen\ndata_Gd4Hf3InIr4\n...
204835    # generated using pymatgen\ndata_Tb3GdMg2Ga3\n...
Name: cif, dtype: object

In [94]:
# Iterate over each row in the DataFrame
for index, row in problem_compounds.iterrows():
    # Use the entry in 'material_id' as the filename
    filename = f"{row['material_id']}.cif"
    
    # The content for the file from the 'cif' column
    content = row['cif']
    
    # Write the content to a file
    with open(filename, 'w') as file:
        file.write(content)

Some old code / markdown is below

There is some evidence for the idea that the issues with the deepmind data are not due to a differrence generally in the data distributions but rather a few bad compounds "knocking" the model off track. 

The first thing that we'll look at is the first batch that threw the error on the deepmind data. It looks like the first occured on batch 82 on epoch 0.

In [1]:
import torch 
import numpy as np
import pandas as pd

In [2]:
#load in batch_82_0_20.pt
batch_82_0_20 = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20_dm/train_errors/batch_82_0_20.pt')

In [67]:
train_df_dm = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20_dm/train.csv')

In [69]:
def unflatten_atom_types(num_atoms, atom_types):
    """
    Converts a flattened list of atom types into a list of lists based on num_atoms.

    :param num_atoms: A list or tensor of the number of atoms in each molecule.
    :param atom_types: A flattened list or tensor of atom types.
    :return: A list of lists, where each sublist contains atom types for each molecule.
    """
    # Convert tensors to lists if they are not already
    if not isinstance(num_atoms, list):
        num_atoms = num_atoms.tolist()
    if not isinstance(atom_types, list):
        atom_types = atom_types.tolist()

    result = []
    start = 0
    for count in num_atoms:
        end = start + count
        result.append(str(atom_types[start:end]))
        start = end

    return result

In [70]:
# Example usage:
num_atoms = batch_82_0_20[0].num_atoms.cpu()
atom_types = batch_82_0_20[0].atom_types.cpu()

unflattened_atom_types = unflatten_atom_types(num_atoms, atom_types)

In [73]:
filtered_df = train_df_dm[train_df_dm['atomic_numbers'].isin(unflattened_atom_types)]

In [74]:
filtered_df

,composition,cif,xrd,xrd_peak_locations,xrd_peak_intensities,atomic_numbers,disc_sim_xrd,material_id,formation_energy_per_atom
521,by_composition/Co1La1P12Ru3.CIF,# generated using pymatgen\ndata_LaCo(P4Ru)3\n...,DiffractionPattern\n$2\Theta$: [15.54596025 15...,"[15.545960245967844, 15.611668250165447, 22.10...","[38.284404233289536, 6.101930511257095, 34.896...","[57, 27, 15, 15, 15, 15, 15, 15, 15, 15, 15, 1...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_521,0.0
657,by_composition/Er9Ge1Lu1Si5.CIF,# generated using pymatgen\ndata_Er9LuSi5Ge\n_...,DiffractionPattern\n$2\Theta$: [12.22637836 12...,"[12.226378364502791, 12.22857042616683, 12.228...","[0.8230563963389224, 3.124881737454629, 2.3299...","[68, 68, 68, 68, 68, 68, 68, 68, 68, 71, 14, 1...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_657,0.0
1123,by_composition/B8Dy2Fe8.CIF,# generated using pymatgen\ndata_Dy(FeB)4\n_sy...,DiffractionPattern\n$2\Theta$: [18.06608025 18...,"[18.066080250030286, 18.070737866133808, 25.65...","[6.339615938123164, 6.305377706406406, 63.6984...","[66, 66, 26, 26, 26, 26, 26, 26, 26, 26, 5, 5,...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_1123,0.0
1523,by_composition/B6Co9In1Lu2.CIF,# generated using pymatgen\ndata_Lu2In(Co3B2)3...,DiffractionPattern\n$2\Theta$: [15.58224075 20...,"[15.58224074691303, 20.847553107964373, 23.299...","[0.7189943311881286, 35.043549080035554, 1.120...","[71, 71, 49, 27, 27, 27, 27, 27, 27, 27, 27, 2...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_1523,0.0
2697,by_composition/Ag2O14S4.CIF,# generated using pymatgen\ndata_AgS2O7\n_symm...,DiffractionPattern\n$2\Theta$: [ 9.84873581 13...,"[9.848735813474764, 13.051250305041222, 16.321...","[8.435102866601754, 17.681058638496403, 100.0,...","[47, 47, 16, 16, 16, 16, 8, 8, 8, 8, 8, 8, 8, ...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_2697,0.0
...,...,...,...,...,...,...,...,...,...
223437,by_composition/Ag2In1Rh1Sm3Tl2.CIF,# generated using pymatgen\ndata_Sm3Tl2InAg2Rh...,DiffractionPattern\n$2\Theta$: [12.76043058 12...,"[12.760430576080722, 12.842147613320591, 12.86...","[1.5915320787070637, 4.046657131843024, 11.893...","[62, 62, 62, 81, 81, 49, 47, 47, 45]",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_223447,0.0
223585,by_composition/Pa2Pd1Pt5.CIF,# generated using pymatgen\ndata_Pa2PdPt5\n_sy...,DiffractionPattern\n$2\Theta$: [17.41134433 17...,"[17.41134432849515, 17.73728512780863, 17.8404...","[0.7335134254184259, 0.6041885458220436, 1.868...","[91, 91, 46, 78, 78, 78, 78, 78]",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_223595,0.0
223642,by_composition/Ce2La1Pt6Si10Sm1.CIF,# generated using pymatgen\ndata_LaCe2Sm(Si5Pt...,DiffractionPattern\n$2\Theta$: [11.66141648 11...,"[11.661416479107588, 11.671385699108525, 15.24...","[0.4236998428582132, 0.5701379980732947, 7.037...","[57, 58, 58, 62, 14, 14, 14, 14, 14, 14, 14, 1...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_223652,0.0
225407,by_composition/La4Lu6Rh1Ru3.CIF,# generated using pymatgen\ndata_La4Lu6Ru3Rh\n...,DiffractionPattern\n$2\Theta$: [11.10603703 12...,"[11.106037033757937, 12.379921459498547, 14.78...","[1.2546072675511295, 0.021971860177929688, 18....","[57, 57, 57, 57, 71, 71, 71, 71, 71, 71, 44, 4...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,train_not_true_mat_id_225417,0.0


Now that we've run a separate training run, we have a new batch of compounds that are causing issues. We can cross-reference to see if we can find a common culprit. 